<a href="https://colab.research.google.com/github/redddddsu/Adaptative-SNN-Based-Triggering/blob/main/Software_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing API/Library


In [1]:
!pip install snntorch

In [91]:
import torch
import torch.nn as nn
import snntorch as snn
import matplotlib.pyplot as plt
import os
import librosa
import numpy as np
from snntorch import surrogate



# Dataset

In [3]:

save_dir = "/content/drive/MyDrive/TUT_dataset"
os.makedirs(save_dir, exist_ok=True)

!wget -c -O /content/tut_source_events.zip \
"https://zenodo.org/records/401395/files/TUT-rare-sound-events-2017-development.source_data_events.zip?download=1"

!unzip -o /content/tut_source_events.zip -d /content/


--2026-10-26 05:23:41--  https://zenodo.org/records/401395/files/TUT-rare-sound-events-2017-development.source_data_events.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.51.60, 188.185.40.132, 188.185.45.63, ...
Connecting to zenodo.org (zenodo.org)|188.185.51.60|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 639119477 (610M) [application/octet-stream]
Saving to: ‘/content/tut_source_events.zip’

/content/tut_source 100%[===================>] 609.51M  4.49MB/s    in 3m 39s  

2026-10-26 05:27:20 (2.79 MB/s) - ‘/content/tut_source_events.zip’ saved [639119477/639119477]

Archive:  /content/tut_source_events.zip
   creating: /content/TUT-rare-sound-events-2017-development/
   creating: /content/TUT-rare-sound-events-2017-development/data/
   creating: /content/TUT-rare-sound-events-2017-development/data/source_data/
   creating: /content/TUT-rare-sound-events-2017-development/data/source_data/events/
   creating: /content/TUT-rare-sound-events-2017

In [96]:
babycry_path = "/content/TUT-rare-sound-events-2017-development/data/source_data/events/babycry"
glassbreak_path = "/content/TUT-rare-sound-events-2017-development/data/source_data/events/glassbreak"
gunshot_path = "/content/TUT-rare-sound-events-2017-development/data/source_data/events/gunshot"

all_paths = {
    "babycry": babycry_path,
    "glassbreak": glassbreak_path,
    "gunshot": gunshot_path
}

recordings = []

for event, path in all_paths.items():
    wav_files = [
        f for f in os.listdir(path)
        if f.lower().endswith(".wav")
    ]

    yaml_files = [
        f for f in os.listdir(path)
        if f.lower().endswith(".yaml")
    ]

    for filename in wav_files:
        recordings.append({
            "event": event,
            "wav_path": os.path.join(path, filename)
        })


# Mel Spectrogem

In [97]:
mel_data = {}

n_mels = 128
window_seconds = 0.064
hop_seconds = 0.032

for recording in recordings:
    event = recording["event"]
    wav_path = recording["wav_path"]

    audio, sr = librosa.load(
        wav_path,
        sr=None,
        mono=True
    )

    window_sample = round(sr * window_seconds)
    hop_sample = round(sr * hop_seconds)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=window_sample,
        win_length=window_sample,
        hop_length=hop_sample,
        n_mels=n_mels,
        fmin=0,
        fmax=sr / 2,
        power=2
    )

    log_mel = librosa.power_to_db(
        mel,
        ref=np.max
    )

    filename = os.path.basename(wav_path)

    mel_data[filename] = {
        "log_mel": log_mel,
        "sr": sr,
        "duration": len(audio) / sr,
        "event": event
    }

# Labeling Frames

In [101]:
import yaml

for recording in recordings:
    wav_path = recording["wav_path"]
    recording["yaml_path"] = os.path.splitext(wav_path)[0] + ".yaml"

event_to_label = {
    "babycry": 1,
    "glassbreak": 2,
    "gunshot": 3
}

for recording in recordings:
    filename = os.path.basename(recording["wav_path"])

    # Use the mel-spectrogram you already calculated
    log_mel = mel_data[filename]["log_mel"]
    sr = mel_data[filename]["sr"]

    # Read the annotation
    with open(recording["yaml_path"], "r") as f:
        annotation = yaml.unsafe_load(f)

    hop_length = round(sr * 0.032)

    # Start every frame as background
    labels = np.zeros(log_mel.shape[1], dtype=np.int64)

    # Mark frames containing the annotated event
    for start, end in annotation["valid_segments"]:
        start_frame = int(np.floor(start * sr / hop_length))
        end_frame = int(np.ceil(end * sr / hop_length))

        start_frame = max(0, min(start_frame, len(labels)))
        end_frame = max(0, min(end_frame, len(labels)))

        labels[start_frame:end_frame] = event_to_label[
            recording["event"]
        ]

    mel_data[filename]["labels"] = labels


Labeling complete!


# SNN Frame


In [109]:
spike_grad = surrogate.fast_sigmoid()

class SNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(128, 128)
        self.lif1 = snn.Leaky(
            beta=0.8,
            threshold=1.0,
            spike_grad=spike_grad,
            reset_mechanism="subtract"
        )

        self.layer2 = nn.Linear(128, 1)
        self.lif2 = snn.Leaky(
            beta=0.8,
            threshold=1.0,
            spike_grad=spike_grad,
            reset_mechanism="subtract"
        )

    def forward(self, x):
        potential1 = self.lif1.init_leaky()
        potential2 = self.lif2.init_leaky()

        current1_all = self.layer1(x)
        spikes = []

        for t in range(x.shape[0]):
            spk1, potential1 = self.lif1(
                current1_all[t], potential1
            )
            current2 = self.layer2(spk1)
            spk2, potential2 = self.lif2(
                current2, potential2
            )
            spikes.append(spk2)

        return torch.stack(spikes)





# Van Rossum Loss

In [110]:
def van_rossum_loss(predict, target, dt=0.032, tau=0.1):
    alpha = torch.exp(torch.tensor(-dt / tau, device=predict.device))

    pred_filter = []
    target_filter = []

    pred_state = torch.zeros_like(predict[0])
    target_state = torch.zeros_like(target[0])

    for t in range(predict.shape[0]):
        pred_state = alpha * pred_state + predict[t]
        target_state = alpha * target_state + target[t]

        pred_filter.append(pred_state)
        target_filter.append(target_state)

    pred_filter = torch.stack(pred_filter)
    target_filter = torch.stack(target_filter)

    loss = (dt / tau) * torch.mean((pred_filter - target_filter) ** 2)

    return loss

# Training

In [112]:
import random

random.seed(42)
torch.manual_seed(42)

items = [
    (filename, data)
    for filename, data in mel_data.items()
    if "labels" in data
]

print("Labeled recordings:", len(items))

assert len(items) >= 2, (
    "Not enough labeled recordings. Make sure your labeling step completed."
)

random.shuffle(items)

split = max(1, int(0.8 * len(items)))
split = min(split, len(items) - 1)

train_items = items[:split]
val_items = items[split:]

print("Training recordings:", len(train_items))
print("Validation recordings:", len(val_items))

Labeled recordings: 160
Training recordings: 128
Validation recordings: 32


In [113]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = SNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def prepare_recording(data):
    log_mel = data["log_mel"].T
    x = np.clip((log_mel + 80.0) / 80.0, 0.0, 1.0)
    y = (data["labels"] > 0).astype(np.float32)
    x = torch.tensor(x, dtype=torch.float32, device=device)
    y = torch.tensor(y, dtype=torch.float32, device=device).unsqueeze(1)

    return x, y


epochs = 3

for epoch in range(epochs):
    model.train()
    random.shuffle(train_items)
    total_loss = 0.0

    for filename, data in train_items:
        x, target = prepare_recording(data)

        optimizer.zero_grad()

        prediction = model(x)
        loss = van_rossum_loss(prediction, target)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_items)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for filename, data in val_items:
            x, target = prepare_recording(data)
            prediction = model(x)
            loss = van_rossum_loss(prediction, target)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_items)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {avg_train_loss:.6f} | "
        f"Validation Loss: {avg_val_loss:.6f}"
    )

Epoch 1/3 | Train Loss: 0.629651 | Validation Loss: 0.584361
Epoch 2/3 | Train Loss: 0.527091 | Validation Loss: 0.608227
Epoch 3/3 | Train Loss: 0.528020 | Validation Loss: 0.633720


# Eval

In [114]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()

all_true = []
all_pred = []

with torch.no_grad():
    for filename, data in val_items:
        x, target = prepare_recording(data)
        prediction = model(x)

        pred = (prediction > 0).int().cpu().numpy().reshape(-1)
        true = (target > 0).int().cpu().numpy().reshape(-1)

        all_true.extend(true.tolist())
        all_pred.extend(pred.tolist())

print("\nClassification report:")
print(classification_report(
    all_true,
    all_pred,
    labels=[0, 1],
    target_names=["Background", "Event"],
    zero_division=0
))

True labels: (array([0, 1]), array([17605,  4757]))
Predicted labels: (array([0, 1]), array([14016,  8346]))

Confusion matrix:
[[12151  5454]
 [ 1865  2892]]

Classification report:
              precision    recall  f1-score   support

  Background       0.87      0.69      0.77     17605
       Event       0.35      0.61      0.44      4757

    accuracy                           0.67     22362
   macro avg       0.61      0.65      0.60     22362
weighted avg       0.76      0.67      0.70     22362



# Result

In [115]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(
    all_true, all_pred, labels=[0, 1]
).ravel()

print("True Positive (TP):", tp)
print("False Positive (FP):", fp)
print("True Negative (TN):", tn)
print("False Negative (FN):", fn)

True Positive (TP): 2892
False Positive (FP): 5454
True Negative (TN): 12151
False Negative (FN): 1865
